<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/July11_Privacy_Layer_Implementation_Arul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Privacy Layer Implementation**

### **Overview**
In this demonstration, we explore how to build a **Privacy Layer** for an AI agent — a fundamental step in ensuring *ethical, safe, and compliant* handling of user data within agentic systems.  
Before any user text is processed by a Large Language Model (LLM), the system applies an **input sanitization layer** that detects and masks personally identifiable information (PII) such as phone numbers, email addresses, or financial identifiers.  

This ensures that **no sensitive user information ever reaches the model context**, reducing risks of data leakage, misuse, or inadvertent memorization.

---

### **Objective**
The goal of this demo is to:
- **Detect and remove private entities** from user queries using regex-based pattern matching.  
- **Apply sanitization before inference**, ensuring models process only anonymized text.  
- **Illustrate ethical data handling practices** in line with Responsible AI and Data Privacy principles.

---

### **What the Demo Implements**

| Layer | Description | Implementation |
|--------|--------------|----------------|
| **Masking Function** | Uses pattern recognition to find and replace sensitive entities (phone, email, card, PAN, Aadhaar) with anonymized tokens. | Implemented as `mask_private_data()` using regular expressions. |
| **Pre-Processing Layer** | Acts as a protective middleware before the model call, sanitizing all incoming inputs. | Implemented as `sanitize_and_respond()` — it cleans the text before invoking the model. |

---

### **Why This Matters**
This privacy layer represents the **ethical foundation of AI governance** — embedding privacy-by-design directly into agent workflows.  
Without such filtering, LLMs could:
- Store or propagate private data across interactions.  
- Expose sensitive user details in future outputs.  
- Violate organizational data handling policies or legal compliance frameworks (GDPR, DPDP, HIPAA, etc.).

By integrating this masking layer, developers demonstrate **proactive adherence to Responsible AI principles** — ensuring that user trust and data safety are preserved at the system level.

---

### **Learning Outcome**
After completing this walkthrough, learners will be able to:
- Design privacy-preserving components in agent pipelines.  
- Understand how **ethical principles translate into technical safeguards**.  
- Extend this approach into larger **Responsible AI architectures** including safety filters, audit logs, and governance dashboards.


In [ ]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully uninstalled openai-2.43.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8


In [ ]:
import re
from langchain_openai import ChatOpenAI
from google.colab import userdata

# Initialize the model (using a lightweight version for demo)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
# ==========================================================
# Masking Function
# ==========================================================
def mask_private_data(text: str) -> str:
    """
    Detects and masks sensitive information like phone numbers,
    emails, credit card numbers, PAN, etc., before LLM processing.
    """
    # Phone numbers (India + international)
    text = re.sub(
        r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "[PHONE]", text)

    # Email addresses
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "[EMAIL]", text, flags=re.IGNORECASE)

    # Credit card numbers (13–19 digits)
    text = re.sub(
        r'\b(?:\d[ -]*?){13,19}\b',
        "[CARD]", text)

    # Indian PAN format
    text = re.sub(r'\b[A-Z]{5}\d{4}[A-Z]\b', "[PAN]", text)

    # Aadhaar-like 12-digit sequences
    text = re.sub(r'\b\d{12}\b', "[ID]", text)

    return text


# ==========================================================
# Pre-processing Layer
# ==========================================================
def sanitize_and_respond(user_input: str) -> str:
    """
    Pre-processing layer that sanitizes all incoming user text
    before passing to the model.
    """
    sanitized = mask_private_data(user_input)
    response = llm.invoke(sanitized)
    return f"Sanitized Input:\n{sanitized}\n\nModel Response:\n{response.content}"


# ==========================================================
# Demo Execution
# ==========================================================
if __name__ == "__main__":
    sample_inputs = [
        "Hi, my phone is 9876543210 and email is ankit@abc.com. What is Responsible AI?",
        "Send report for card 1234-5678-9876-5432 and PAN ABCDE1234F",
        "Aadhaar 999988887777 linked to my account; please explain privacy laws."
    ]

    for i, text in enumerate(sample_inputs, 1):
        print(f"\n--- Example {i} ---")
        print(sanitize_and_respond(text))


--- Example 1 ---
Sanitized Input:
Hi, my phone is [PHONE] and email is [EMAIL]. What is Responsible AI?

Model Response:
Responsible AI refers to the development and deployment of artificial intelligence systems in a manner that is ethical, transparent, and accountable. It encompasses a set of principles and practices aimed at ensuring that AI technologies are used in ways that are beneficial to society and do not cause harm. Key aspects of Responsible AI include:

1. **Fairness**: Ensuring that AI systems do not perpetuate biases or discrimination against individuals or groups based on race, gender, age, or other characteristics.

2. **Transparency**: Making AI systems understandable and explainable to users, allowing them to comprehend how decisions are made.

3. **Accountability**: Establishing clear lines of responsibility for the outcomes of AI systems, including mechanisms for redress in case of harm or error.

4. **Privacy**: Protecting individuals' data and ensuring that AI s